# Vision 실습 문제 해설 (Solution)

각 문제마다 **핵심 개념 설명 → 코드 정답 + 풍부한 주석** 순으로 구성되어 있습니다.

## 문제 분류 (출제 기준)
- **데이터 구성 & Library 활용**: 문제 1, 14
- **평가 Metric 이해**: 문제 2, 3
- **학습 원리**: 문제 4, 5
- **모델 원리 (ViT)**: 문제 6~13
- **모델 원리 (DETR)**: 문제 15~18
- **모델 원리 (U-Net)**: 문제 19~23

## 문제 1. (a-1) CIFAR-10 학습용 Transform 구성
### 핵심 개념: torchvision.transforms 파이프라인

- **정의(definition):** `T.Compose([...])` 는 여러 이미지 변환을 순차적으로 실행합니다. 학습용 파이프라인은 보통 `증강(Augmentation) → Tensor 변환 → Normalize` 순서로 구성합니다.
- **원리:**
    - `T.RandomCrop(32, padding=4)`: 4픽셀 zero-padding 후 32×32 랜덤 크롭 → 위치 invariance 유도.
    - `T.RandomHorizontalFlip()`: 50% 확률로 좌우 반전 → 수평 대칭 데이터 증강.
    - `T.ToTensor()`: PIL(H,W,C)·0~255 → Tensor(C,H,W)·0~1 로 변환.
    - `T.Normalize(mean, std)`: `(x - mean) / std` 로 채널별 정규화. 반드시 `ToTensor` 다음에 와야 tensor에 대해 연산됩니다.
- **예시:** CIFAR-10 평균·표준편차를 사용하면 값 범위가 대략 `-2.5 ~ 2.5`가 되어 학습 초기 기울기 분포가 안정됩니다.
- **주의:** 테스트셋에는 증강을 넣지 않고 `ToTensor + Normalize`만 적용합니다.

In [ ]:
# [코드]
import torchvision.transforms as T

CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2023, 0.1994, 0.2010)

train_tfms = T.Compose([
    T.RandomCrop(32, padding=4),                 # 4픽셀 패딩 후 32x32 랜덤 크롭 (공간 증강)
    T.RandomHorizontalFlip(),                    # 좌우 반전 (대칭 증강)
    T.ToTensor(),                                # PIL -> Tensor (C,H,W), [0,1]
    T.Normalize(CIFAR10_MEAN, CIFAR10_STD),      # (x - mean)/std 정규화
])

## 문제 2. (a-2) Top-1 Accuracy 계산
### 핵심 개념: argmax 기반 분류 정확도

- **정의(definition):** 모델이 출력한 로짓(logits) 중 가장 큰 값의 클래스를 예측값으로 삼고, 정답(label)과 일치한 비율을 구한 지표가 **Top-1 Accuracy**입니다.
- **원리:** `logits.argmax(dim=1)` 은 클래스 축에서 최댓값의 인덱스를 반환 → `(B,)` 예측 클래스 텐서. `(preds == targets).float().mean()` 으로 맞춘 비율을 스칼라로 얻습니다.
- **예시:** `logits = [[0.1, 2.3, 0.5], [1.0, 0.2, 3.1]]` → argmax = `[1, 2]`. 정답 `[1, 0]` → 정확도 0.5.

In [ ]:
# [코드]
def accuracy_top1(logits, targets):
    preds = logits.argmax(dim=1)                  # (B,) 예측 클래스
    return (preds == targets).float().mean().item()  # 맞춘 비율 (Python scalar)

## 문제 3. (a-3) Confusion Matrix 집계
### 핵심 개념: Confusion Matrix의 인덱싱 규칙

- **정의(definition):** Confusion Matrix `CM`은 행(row)이 정답(Ground Truth), 열(column)이 예측(Prediction)인 `C×C` 카운트 행렬입니다. 대각 원소는 맞춘 개수, 그 외는 혼동된 개수입니다.
- **원리:** 한 샘플 `(t, p)`가 들어오면 `CM[t, p] += 1`. 텐서 원소는 정수로 캐스팅해 인덱스로 사용합니다.
- **예시:** class별 정확도는 `CM.diagonal() / CM.sum(dim=1)` 로 구합니다. 분모가 0인 클래스는 NaN 방지 위해 `clamp(min=1.0)` 합니다.

In [ ]:
# [코드]
import torch

def confusion_and_perclass(preds, labels, num_classes):
    cm = torch.zeros((num_classes, num_classes), dtype=torch.int64)
    for t, p in zip(labels, preds):
        # 행=정답(t), 열=예측(p) 위치의 카운트를 1 증가
        cm[int(t), int(p)] += 1
    # 클래스별 정확도: 대각 / 해당 클래스 총 개수(행 합)
    per_class_acc = cm.diagonal().float() / torch.clamp(cm.sum(dim=1).float(), min=1.0)
    return cm, per_class_acc

## 문제 4. (a-4) 학습 루프의 핵심 3단계
### 핵심 개념: zero_grad → backward → step

- **정의(definition):** PyTorch 학습은 매 배치마다 **gradient 초기화 → 역전파 → 파라미터 갱신** 3단계를 반복합니다.
- **원리:**
    1. `optimizer.zero_grad()`: 이전 배치에서 누적된 gradient를 0으로 초기화. PyTorch는 gradient를 **누적(accumulate)** 하기 때문에 필수.
    2. `loss.backward()`: 자동미분으로 각 파라미터에 대한 `∂loss/∂w` 계산.
    3. `optimizer.step()`: 계산된 gradient로 optimizer 규칙(SGD, Adam 등)에 따라 파라미터 갱신.
- **예시:** 순서를 지키지 않거나 `zero_grad()`를 빼먹으면 gradient가 무한 누적되어 학습이 발산합니다.

In [ ]:
# [코드]
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()                         # Dropout/BN을 학습 모드로
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()             # 1) 이전 gradient 초기화
        logits = model(images)            # forward
        loss = criterion(logits, labels)  # loss 계산
        loss.backward()                   # 2) 역전파: gradient 계산
        optimizer.step()                  # 3) 파라미터 갱신

## 문제 5. (a-5) SGD + Momentum Optimizer
### 핵심 개념: SGD with Momentum + Weight Decay

- **정의(definition):** 기본 SGD `w ← w − lr·g` 에 **Momentum**(과거 gradient의 지수이동평균 `v ← μ·v + g`, `w ← w − lr·v`)과 **Weight Decay**(L2 정규화, `− lr·λ·w`)를 결합한 optimizer입니다.
- **원리:**
    - Momentum(보통 0.9): 수렴 가속 + 진동 억제.
    - Weight Decay(보통 `5e-4`): 가중치 크기를 제한해 과적합 완화.
- **예시:** ResNet18 + CIFAR-10 표준 레시피는 `lr=0.1, momentum=0.9, weight_decay=5e-4` + CosineAnnealingLR.

In [ ]:
# [코드]
import torch
import torch.nn as nn

criterion = nn.CrossEntropyLoss()  # 분류 표준 loss (softmax + NLL 결합)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.1,
    momentum=0.9,       # 과거 gradient 관성
    weight_decay=5e-4,  # L2 정규화
)

## 문제 6. (b-1) ViT PreNorm (LayerNorm)
### 핵심 개념: Pre-LayerNorm Transformer

- **정의(definition):** 원 Transformer는 Post-LN(`x + SubLayer(x) → LN`)이지만, ViT·최근 구현은 **Pre-LN**(`x + SubLayer(LN(x))`)을 주로 사용합니다.
- **원리:** 입력 토큰을 **마지막 차원(embedding dim)** 기준으로 `LayerNorm`하면 각 토큰별 평균 0, 분산 1로 정규화되어 학습이 안정화되고 깊은 네트워크에서도 기울기 소실/폭주가 완화됩니다. `nn.LayerNorm(dim)` 은 마지막 축의 크기가 `dim`인 축을 정규화합니다.
- **예시:** `(B, N, dim)` 텐서 → `LayerNorm(dim)` 적용 시 각 토큰(N 위치 각각)이 개별 정규화됨.

In [ ]:
# [코드]
import torch.nn as nn

class PreNorm(nn.Module):
    def __init__(self, dim, fn):
        super().__init__()
        # 토큰 임베딩 차원(dim)을 기준으로 LayerNorm
        self.norm = nn.LayerNorm(dim)
        self.fn = fn

    def forward(self, x, **kwargs):
        return self.fn(self.norm(x), **kwargs)

## 문제 7. (b-2) ViT FeedForward (Linear → GELU → Linear)
### 핵심 개념: Transformer FFN(MLP) 블록

- **정의(definition):** Self-Attention이 토큰 간 정보를 섞는다면, FFN은 각 토큰 **내부의 채널 방향**으로 비선형 변환을 수행합니다. 표준 구조는 `Linear(dim → hidden_dim) → GELU → Dropout → Linear(hidden_dim → dim) → Dropout` 입니다.
- **원리:**
    - 중간 차원을 `hidden_dim`(보통 `4 × dim`)으로 넓혀 더 풍부한 비선형 조합을 학습.
    - **GELU** 활성화는 ReLU와 달리 부드러운 곡선을 가져 Transformer 계열에서 성능이 더 좋습니다.
- **예시:** `dim=64, hidden_dim=128` → FFN 파라미터 수는 `64·128 + 128·64 ≈ 16K`.

In [ ]:
# [코드]
import torch.nn as nn

class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim, dropout=0.):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),      # 채널 확장 (dim -> hidden_dim)
            nn.GELU(),                        # 부드러운 비선형 활성화
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),       # 원래 차원으로 복원 (hidden -> dim)
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

## 문제 8. (b-3) Multi-Head Attention inner_dim 계산
### 핵심 개념: Multi-Head 분할 차원

- **정의(definition):** Multi-Head Attention은 하나의 attention을 `heads` 개의 서브 공간에서 병렬로 수행합니다. 각 head의 차원은 `dim_head`이고, 이들을 concat한 전체 차원 `inner_dim = heads × dim_head` 입니다.
- **원리:** 입력(`dim`) → `to_qkv`로 `inner_dim × 3` 를 만든 뒤 Q/K/V로 split. 이후 `rearrange('b n (h d) -> b h n d')` 로 head 축을 분리합니다.
- **예시:** `heads=4, dim_head=64` → `inner_dim = 256`. 모델 hidden `dim=64`라면 `to_qkv`가 `Linear(64, 768)`이 됩니다.

In [ ]:
# [코드]
import torch.nn as nn

class AttentionInit(nn.Module):
    def __init__(self, dim, heads=4, dim_head=64):
        super().__init__()
        # 전체 Q/K/V 차원 = head 수 × 각 head 차원
        inner_dim = dim_head * heads
        self.to_qkv = nn.Linear(dim, inner_dim * 3, bias=False)
        self.to_out = nn.Linear(inner_dim, dim)

## 문제 9. (b-4) Scaled Dot-Product Attention
### 핵심 개념: Q·Kᵀ / √dₖ + softmax

- **정의(definition):** `Attn(Q,K,V) = softmax(Q·Kᵀ / √dₖ) · V`. Q가 각 K와의 내적으로 유사도를 구하고 softmax 확률로 변환한 뒤 V의 가중합을 얻습니다.
- **원리:**
    - `einsum('b h i d, b h j d -> b h i j', q, k)`: head별로 쿼리 i와 키 j의 내적 (`(B, H, N, N)`).
    - 스케일 `1/√dₖ`: 차원이 커질수록 내적 분산이 커져 softmax가 포화되는 것을 방지.
    - `softmax(dim=-1)`: 각 쿼리별 키 방향으로 확률화(합=1).
- **예시:** `dₖ=64` → scale `0.125`. 스케일이 없으면 초기 학습이 불안정해짐.

In [ ]:
# [코드]
import torch
from torch import nn, einsum
from einops import rearrange

class Attention(nn.Module):
    def __init__(self, dim, heads=4, dim_head=64):
        super().__init__()
        inner_dim = dim_head * heads
        self.heads = heads
        self.scale = dim_head ** -0.5   # 1/√dₖ
        self.attend = nn.Softmax(dim=-1)
        self.to_qkv = nn.Linear(dim, inner_dim * 3, bias=False)
        self.to_out = nn.Linear(inner_dim, dim)

    def forward(self, x):
        b, n, _, h = *x.shape, self.heads
        qkv = self.to_qkv(x).chunk(3, dim=-1)
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h=h), qkv)

        # Q·Kᵀ 유사도 계산 (scaled dot-product)
        dots = einsum('b h i d, b h j d -> b h i j', q, k) * self.scale
        # softmax 로 attention 확률 변환
        attn = self.attend(dots)

        out = einsum('b h i j, b h j d -> b h i d', attn, v)
        out = rearrange(out, 'b h n d -> b n (h d)')
        return self.to_out(out)

## 문제 10. (b-5) Transformer Block 구성
### 핵심 개념: Encoder 블록 한 층의 구조

- **정의(definition):** ViT 한 층은 **PreNorm + Attention** 서브블록과 **PreNorm + FeedForward** 서브블록의 쌍으로 이루어지며, 각 서브블록은 residual로 감싸집니다.
- **원리:** `layers.append([PreNorm(dim, Attention(...)), PreNorm(dim, FeedForward(...))])` 패턴을 `depth`만큼 반복. forward에서 `x = attn(x) + x`, `x = ff(x) + x` 로 residual 연결.
- **예시:** `depth=6` 이면 Attention과 FFN이 각각 6번씩 쌓인 총 12개 LayerNorm이 존재.

In [ ]:
# [코드]
import torch.nn as nn

class Transformer(nn.Module):
    def __init__(self, dim, depth, heads, dim_head, mlp_dim, dropout=0.):
        super().__init__()
        self.layers = nn.ModuleList([])
        for _ in range(depth):
            self.layers.append(nn.ModuleList([
                # 서브블록 1: (PreNorm + Multi-Head Attention)
                PreNorm(dim, Attention(dim, heads=heads, dim_head=dim_head, dropout=dropout)),
                # 서브블록 2: (PreNorm + FeedForward)
                PreNorm(dim, FeedForward(dim, mlp_dim, dropout=dropout)),
            ]))

    def forward(self, x):
        for attn, ff in self.layers:
            x = attn(x) + x   # residual connection
            x = ff(x) + x     # residual connection
        return x

## 문제 11. (b-6) ViT 패치 임베딩 (einops Rearrange)
### 핵심 개념: 이미지의 패치 토큰화

- **정의(definition):** ViT는 이미지를 `P×P` 패치로 자른 뒤 각 패치를 1D 벡터로 펼쳐 토큰 시퀀스로 취급합니다. einops 패턴 `"b c (h p1) (w p2) -> b (h w) (p1 p2 c)"`가 이 변환을 깔끔히 표현합니다.
- **원리:** `(h p1)`은 원본 H, `(w p2)`는 원본 W를 의미. 결과 shape는 `(B, N=h·w, P·P·C)`. 이후 `Linear(patch_dim, dim)`으로 Transformer hidden 차원으로 투영합니다.
- **예시:** CIFAR-10 32×32·3채널 + 패치 4×4 → N=64, patch_dim=48. `Linear(48, 64)` → `(B, 64, 64)`.

In [ ]:
# [코드]
from einops.layers.torch import Rearrange
import torch.nn as nn

patch_height = 4
patch_width = 4
patch_dim = 3 * patch_height * patch_width
dim = 64

to_patch_embedding = nn.Sequential(
    # h,w: 패치 그리드 개수 / p1,p2: 패치 내부 픽셀
    # (B, C, H, W) -> (B, N=h·w, P·P·C)
    Rearrange("b c (h p1) (w p2) -> b (h w) (p1 p2 c)",
              p1=patch_height, p2=patch_width),
    nn.Linear(patch_dim, dim),  # 패치 벡터 -> hidden dim 투영
)

## 문제 12. (b-7) ViT CLS 토큰 결합 & 위치 임베딩
### 핵심 개념: [CLS] 토큰과 Positional Embedding

- **정의(definition):**
    - **[CLS] 토큰**: 이미지 전체 표현을 담는 학습 가능한 특수 토큰 1개. 토큰 시퀀스 맨 앞에 추가하고 최종 분류에 `x[:, 0]`만 사용.
    - **Positional Embedding**: Self-Attention은 순서 정보를 모르므로, 각 위치마다 학습 벡터를 더해 공간 정보를 주입.
- **원리:**
    - `repeat(self.cls_token, '1 1 d -> b 1 d', b=b)` 로 CLS를 배치 크기만큼 복제.
    - `torch.cat((cls_tokens, x), dim=1)` 로 시퀀스 맨 앞에 concat → `(B, N+1, dim)`.
    - `x + self.pos_embedding[:, :(n+1)]` 로 위치 임베딩을 broadcast 가산.
- **예시:** N=64 패치 → CLS 추가 후 65 토큰 → `pos_embedding.shape = (1, 65, dim)`.

In [ ]:
# [코드]
import torch
from torch import nn
from einops import repeat

class ViTForward(nn.Module):
    def __init__(self, num_patches, dim):
        super().__init__()
        self.cls_token = nn.Parameter(torch.randn(1, 1, dim))
        self.pos_embedding = nn.Parameter(torch.randn(1, num_patches + 1, dim))

    def forward(self, x):
        b, n, _ = x.shape
        # (1,1,d) -> (b,1,d) 로 배치 크기만큼 CLS 토큰 복제
        cls_tokens = repeat(self.cls_token, '1 1 d -> b 1 d', b=b)
        # 시퀀스 맨 앞에 CLS concat -> (B, N+1, dim)
        x = torch.cat((cls_tokens, x), dim=1)
        # 위치 임베딩 더하기 (N+1 길이만큼 슬라이스)
        x = x + self.pos_embedding[:, :(n + 1)]
        return x

## 문제 13. (b-8) ViT 분류 표현 추출: CLS vs mean pooling
### 핵심 개념: 시퀀스 풀링 방식 선택

- **정의(definition):** Transformer 출력 `(B, N+1, dim)`에서 분류용 단일 벡터를 고르는 방법은 크게 두 가지입니다.
    - `pool == "cls"`: 맨 앞 CLS 토큰 `x[:, 0]`.
    - `pool == "mean"`: 모든 토큰 평균 `x.mean(dim=1)`.
- **원리:** CLS 방식은 학습되면서 '요약자' 역할을 맡고, mean pool은 모든 토큰에 동등 가중치를 부여합니다. 데이터·세팅에 따라 선택.
- **예시:** 원 ViT 논문은 CLS, BEiT/Swin 일부는 mean/GAP 방식을 사용.

In [ ]:
# [코드]
def select_feature(x, pool):
    # "cls": 0번 토큰만 사용 / "mean": 전체 토큰 평균
    x = x[:, 0] if pool == "cls" else x.mean(dim=1)
    return x

## 문제 14. (c-1) DETR 입력 전처리 파이프라인
### 핵심 개념: 사전학습 모델과 전처리 정합성

- **정의(definition):** 사전학습 모델을 추론에 쓰려면 학습 시 사용한 전처리(resize, tensor 변환, 정규화)를 그대로 맞춰야 합니다.
- **원리:** DETR backbone은 ImageNet으로 사전학습된 ResNet이므로 `mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)` 로 정규화. 공식 데모는 짧은 변을 800픽셀로 `Resize` 합니다. 순서는 반드시 `Resize → ToTensor → Normalize`.
- **예시:** 정규화를 빼면 객체 탐지 신뢰도가 급락하고 엉뚱한 박스가 나옵니다.

In [ ]:
# [코드]
import torchvision.transforms as T

transform = T.Compose([
    T.Resize(800),                      # 짧은 변을 800px로 리사이즈 (DETR 공식 설정)
    T.ToTensor(),                       # PIL -> Tensor (C,H,W), [0,1]
    T.Normalize([0.485, 0.456, 0.406],
                [0.229, 0.224, 0.225]), # ImageNet mean/std 정규화
])

## 문제 15. (c-2) 박스 좌표 변환 (cx,cy,w,h) → (xmin,ymin,xmax,ymax)
### 핵심 개념: 바운딩 박스 표현 변환

- **정의(definition):**
    - `cxcywh`: 중심 좌표와 너비/높이. DETR·YOLO 등 모델 내부 표현.
    - `xyxy`: 좌상단·우하단 꼭짓점. 시각화·NMS·IoU 계산에서 주로 사용.
- **원리:** 중심과 크기로부터 꼭짓점 계산: `xmin = cx − w/2`, `ymin = cy − h/2`, `xmax = cx + w/2`, `ymax = cy + h/2`. 계산된 4개 텐서를 `torch.stack(b, dim=1)` 로 `(N, 4)` 형태로 결합합니다.
- **예시:** 중심 `(100,120), w=40, h=30` → `(80, 105, 120, 135)`.

In [ ]:
# [코드]
import torch

def box_cxcywh_to_xyxy(x):
    x_c, y_c, w, h = x.unbind(1)
    b = [(x_c - 0.5 * w), (y_c - 0.5 * h),
         (x_c + 0.5 * w), (y_c + 0.5 * h)]
    # 4개의 (N,) 벡터를 dim=1 방향으로 쌓아 (N, 4) 로 결합
    return torch.stack(b, dim=1)

## 문제 16. (c-3) 정규화 박스를 픽셀 좌표로 복원
### 핵심 개념: 이미지 크기 복원 스케일링

- **정의(definition):** DETR은 박스를 이미지 크기로 `0~1` 정규화된 상태로 출력하므로, 시각화/후처리를 위해 원본 크기의 `(W, H, W, H)` 벡터로 곱해 픽셀 단위로 되돌립니다.
- **원리:** PIL 이미지의 `size` 속성은 `(W, H)` 순서임을 주의. xyxy 박스는 `x, y, x, y` 순서이므로 곱하는 스케일도 `(W, H, W, H)`.
- **예시:** 정규화 박스 `(0.1, 0.2, 0.5, 0.8)` × `(640, 480, 640, 480)` → `(64, 96, 320, 384)` 픽셀.

In [ ]:
# [코드]
import torch

def rescale_bboxes(out_bbox, size):
    img_w, img_h = size  # PIL size = (W, H)
    b = box_cxcywh_to_xyxy(out_bbox)
    # (W,H,W,H) 순서로 곱해 정규화 박스를 픽셀 좌표로 변환
    b = b * torch.tensor([img_w, img_h, img_w, img_h], dtype=torch.float32)
    return b

## 문제 17. (c-4) DETR 예측 신뢰도 필터링
### 핵심 개념: set prediction과 no-object 클래스

- **정의(definition):** DETR은 NMS 없이 고정 개수의 object query가 각각 `(C+1)` 로짓과 박스를 예측합니다. **+1 은 "no-object"** 클래스로, 실제 객체가 없는 query에 배정.
- **원리:**
    - `softmax(-1)`: 마지막 축(클래스)에서 확률화.
    - `[0, :, :-1]`: 배치 0번 + 마지막(no-object) 클래스 제외 → 실제 객체 클래스 확률.
    - `probas.max(-1).values > 0.9`: 각 query의 최대 score가 임계값 0.9 초과인 경우만 True.
- **예시:** query 100개 중 10개만 임계값을 넘으면 10개의 박스가 남습니다. DETR이 NMS 없이 동작할 수 있는 이유.

In [ ]:
# [코드]
# pred_logits: (1, num_queries, num_classes+1)
# softmax(-1) 후 마지막 no-object 클래스 제거 -> (num_queries, num_classes)
probas = outputs['pred_logits'].softmax(-1)[0, :, :-1]

# 각 query 의 최대 클래스 score 가 0.9 초과인 경우만 True 마스크
keep = probas.max(-1).values > 0.9

## 문제 18. (c-5) DETR keep 마스크로 박스 선택
### 핵심 개념: Bool 마스크 인덱싱

- **정의(definition):** PyTorch에서 bool 텐서 `keep`을 인덱스로 넣으면 `True`인 위치만 선택됩니다 (`boolean masking`).
- **원리:** `outputs['pred_boxes']` shape `(1, num_queries, 4)` → `[0, keep]` → `(num_kept, 4)`. 이후 `rescale_bboxes`로 픽셀 좌표로 변환.
- **예시:** `keep` True 10개 → 최종 박스 10개가 시각화 대상.

In [ ]:
# [코드]
# 배치 0 + bool 마스크로 살아남은 query의 박스만 선택 후 픽셀 단위로 변환
bboxes_scaled = rescale_bboxes(outputs['pred_boxes'][0, keep], im.size)

## 문제 19. (d-1) U-Net DoubleConv 블록
### 핵심 개념: (Conv3x3 → ReLU) × 2 블록

- **정의(definition):** U-Net의 기본 블록은 `Conv3x3 → ReLU → Conv3x3 → ReLU` 구조입니다. `padding=1` 로 공간 해상도 `(H, W)`가 유지됩니다.
- **원리:** 3×3 Conv 2번 = 5×5 receptive field와 동등하지만 파라미터는 적고 비선형성이 2번 들어가 표현력이 증가. BN 유무는 구현마다 다르지만 원 U-Net/본 소스는 BN 없이 `bias=False` + ReLU를 씁니다.
- **예시:** `(B, 64, H, W) → DoubleConv(64, 128) → (B, 128, H, W)`.

In [ ]:
# [코드]
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if mid_channels is None:
            mid_channels = out_channels
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),  # 1차 3x3 Conv: padding=1로 해상도(H, W) 유지
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),  # 2차 3x3 Conv: 최종 출력 채널(out_channels)로 매핑
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

## 문제 20. (d-2) U-Net Down 블록 (MaxPool)
### 핵심 개념: Downsampling = MaxPool + DoubleConv

- **정의(definition):** U-Net 인코더는 각 단계에서 `MaxPool2d(2)` 로 해상도를 절반으로 줄인 뒤 `DoubleConv`로 채널을 확장합니다.
- **원리:** MaxPool은 학습 파라미터 없이 가장 강한 신호만 남겨 translation invariance·연산량 감소·수용영역(receptive field) 확장을 동시에 달성합니다.
- **예시:** `(B, 64, H, W) → MaxPool → (B, 64, H/2, W/2) → DoubleConv(64, 128) → (B, 128, H/2, W/2)`.

In [ ]:
# [코드]
import torch.nn as nn

class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # 해상도를 절반으로 줄이는 MaxPool
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x):
        x = self.pool(x)    # (B, C_in, H/2, W/2)
        x = self.conv(x)    # (B, C_out, H/2, W/2)
        return x

## 문제 21. (d-3) U-Net Up 블록 (Upsample / ConvTranspose2d)
### 핵심 개념: 업샘플링 방식 선택

- **정의(definition):** 디코더는 해상도를 2배로 키우는데 두 가지 방법을 쓸 수 있습니다.
    - **bilinear=True**: `nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)` → 파라미터 없는 이중선형 보간. 채널 유지. 이후 DoubleConv에서 `mid_channels=in_channels//2`로 채널을 절반으로 줄입니다.
    - **bilinear=False**: `nn.ConvTranspose2d(C, C/2, kernel_size=2, stride=2)` → 업샘플 과정 자체를 학습. 채널도 이 레이어가 절반으로 줄여줍니다.
- **원리:** bilinear는 파라미터 절약·artefact 적음, ConvTranspose는 표현력이 높지만 `checkerboard` artefact가 나올 수 있음.
- **예시:** `(B, 1024, 8, 8) → ConvTranspose2d → (B, 512, 16, 16)`.

In [ ]:
# [코드]
import torch.nn as nn

class Up(nn.Module):
    def __init__(self, in_channels, out_channels, bilinear=False):
        super().__init__()
        if bilinear:
            # 파라미터 없는 bilinear 업샘플 (채널 유지)
            self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, mid_channels=in_channels // 2)
        else:
            # 학습 가능한 deconv (채널도 절반으로 축소)
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

## 문제 22. (d-4) U-Net Skip Connection (채널 방향 concat)
### 핵심 개념: Skip 연결로 공간 정보 복원

- **정의(definition):** U-Net의 Skip은 인코더에서 저장한 고해상도 feature `x2`를, 업샘플된 디코더 feature `x1`과 **채널(dim=1)** 방향으로 concat 하는 것입니다.
- **원리:** 인코더는 반복 pooling으로 경계 정보를 잃기 때문에, 사라지기 전의 고해상도 특징을 그대로 가져와 정확한 localization을 가능하게 합니다. 합친 뒤 DoubleConv로 채널을 정제·축소.
- **예시:** `x1=(B,256,32,32)`, `x2=(B,256,32,32)` → `cat(dim=1)` → `(B,512,32,32)` → `DoubleConv(512,256)` → `(B,256,32,32)`.

In [ ]:
# [코드]
import torch
import torch.nn as nn
import torch.nn.functional as F

class UpForward(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        # x1: 디코더 (저해상도) / x2: 인코더 skip (고해상도)
        x1 = self.up(x1)  # 해상도를 skip과 맞춤

        # 홀수 해상도 등 1~2픽셀 편차 보정
        diff_y = x2.size(2) - x1.size(2)
        diff_x = x2.size(3) - x1.size(3)
        x1 = F.pad(x1, [diff_x // 2, diff_x - diff_x // 2,
                        diff_y // 2, diff_y - diff_y // 2])

        # 채널 축(dim=1)으로 concat -> (B, C_skip + C_up, H, W)
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

## 문제 23. (d-5) U-Net 전체 구조
### 핵심 개념: 대칭 Encoder-Decoder + Skip Connections

- **정의(definition):** U-Net은 인코더(Down×4) → 보틀넥 → 디코더(Up×4)의 대칭 구조이며, 각 Up 블록은 **같은 레벨 인코더 출력**을 skip으로 받습니다.
- **원리:**
    - `self.inc = DoubleConv(n_channels, 64)`: 입력 채널 수는 흑백=1, RGB=3 등 데이터에 따라 변하므로 `n_channels`로 전달.
    - 디코더 호출 순서: `up1(x5, x4)` → `up2(x, x3)` → `up3(x, x2)` → `up4(x, x1)` (저해상도 → 고해상도 skip 순서).
- **예시:** 입력 `(B, 1, 128, 128)` → 인코더 내려가며 `x1~x5` 저장 → 디코더에서 역순으로 skip 결합 → 출력 `(B, n_classes, 128, 128)`.

In [ ]:
# [코드]
import torch.nn as nn

class UNet(nn.Module):
    def __init__(self, n_channels, n_classes, bilinear=False):
        super().__init__()
        # 입력 stem: 입력 채널은 데이터에 따라 결정되므로 n_channels 를 그대로 사용
        self.inc = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, n_classes, kernel_size=1)

    def forward(self, x):
        # Encoder: 각 단계 출력을 skip용으로 저장
        x1 = self.inc(x)       # (B, 64,   H,    W)
        x2 = self.down1(x1)    # (B, 128,  H/2,  W/2)
        x3 = self.down2(x2)    # (B, 256,  H/4,  W/4)
        x4 = self.down3(x3)    # (B, 512,  H/8,  W/8)
        x5 = self.down4(x4)    # (B, 1024, H/16, W/16) 보틀넥

        # Decoder: (디코더 feature, 같은 레벨 인코더 skip) 순으로 전달
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.outc(x)